# 🚀 Phase 2 : Modèles Basés sur le Contenu (Content-Based Filtering)

## 🎯 Objectif du Notebook
L'objectif de cette étape est de construire un moteur de recommandation qui s'appuie exclusivement sur le **sens sémantique** des articles. 
Contrairement au filtrage collaboratif, cet algorithme ne regarde pas les interactions des autres utilisateurs. Il analyse les "Embeddings" (le résumé mathématique) du dernier article lu par l'utilisateur, et cherche les articles les plus proches sémantiquement.

L'immense avantage de cette approche est qu'elle **résout nativement le problème du Cold-Start des articles** : même si un article vient d'être publié il y a 10 secondes et n'a aucun clic, le modèle Content-Based peut déjà le recommander !

## 🧠 Distances Mathématiques Mises à l'Épreuve
Afin d'optimiser notre moteur, nous allons comparer deux méthodes de calcul de distance pour mesurer la similarité entre les articles :

1. **La Similarité Cosinus (Cosine Similarity) :** Mesure l'angle entre deux vecteurs (standard absolu en NLP).
2. **La Distance Euclidienne (L2 Distance) :** Mesure la distance spatiale directe entre deux vecteurs.

Nous utiliserons la version compressée de nos embeddings (`articles_embeddings_pca.pickle` - 50 dimensions) générée lors de l'EDA afin de garantir des temps de calcul (latence) compatibles avec les exigences du Cloud Serverless.

## 📏 Métriques d'Évaluation (Critères d'acceptation)
Pour évaluer ce modèle et le comparer équitablement à l'ALS, nous appliquerons le même banc d'essai de validation (*Leave-One-Out Chronologique*) :

| Catégorie | Métrique | Explication |
| :--- | :--- | :--- |
| **Pertinence Métier** | **Hit Ratio @ 5** | Le dernier clic caché de l'utilisateur figure-t-il dans le Top 5 recommandé ? |
| **Précision du Tri** | **MRR @ 5** | À quel rang précis (1er, 2e...) l'article caché a-t-il été proposé ? (Score plus élevé si 1er). |
| **Diversité** | **Coverage** | Pourcentage total d'articles du catalogue qui ont été recommandés (évite l'effet de bulle). |
| **Temps de Réponse** | **Latence (ms)** | Temps pris par l'algorithme pour calculer un Top 5 (doit concurrencer les 3ms de l'ALS). |
| **Empreinte Mémoire** | **Peak RSS / RAM** | Poids des Embeddings et de la matrice à charger dans la RAM de l'Azure Function. |

In [1]:
# Import des packages et mounting drive

# Installation de scikit-learn (pour les calculs de distance)
!pip install scikit-learn

import pandas as pd
import numpy as np
import pickle
import time
import os
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

# Connexion au Google Drive
from google.colab import drive
drive.mount('/content/drive')

# --- MODIFIE CES CHEMINS SELON TON DRIVE ---
DATA_DIR = "/content/drive/MyDrive/OPENCLASSROOMS/Projets/P10/Data"
GENERATED_DIR = "/content/drive/MyDrive/OPENCLASSROOMS/Projets/P10/Generated"

print("📂 Chargement des données pour le Content-Based...")

# 1. Chargement des Embeddings compressés (PCA)
with open(os.path.join(GENERATED_DIR, "articles_embeddings_pca.pickle"), "rb") as f:
    embeddings_pca = pickle.load(f)

# 2. Chargement des Metadata (Utile pour vérifier la catégorie des articles recommandés)
articles_df = pd.read_csv(os.path.join(DATA_DIR, "articles_metadata.csv"))

print(f"✅ Embeddings chargés. Format : {embeddings_pca.shape} (Articles x Dimensions)")
print(f"✅ Metadata chargées. Total articles : {len(articles_df)}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Chargement des données pour le Content-Based...
✅ Embeddings chargés. Format : (364047, 52) (Articles x Dimensions)
✅ Metadata chargées. Total articles : 364047


In [2]:
# ==========================================
# MOTEUR DE RECOMMANDATION CONTENT-BASED
# ==========================================

print("🎯 Création du moteur de recommandation...")

def recommend_content_based(article_id, embeddings, top_n=5, distance_metric='cosine'):
    """
    Recommande les 'top_n' articles les plus similaires à 'article_id'.
    Prend en charge les métriques 'cosine' et 'euclidean'.
    Retourne les IDs, les scores de similarité, et le temps de calcul.
    """
    # 1. On isole le vecteur mathématique de l'article lu
    target_vector = embeddings[article_id].reshape(1, -1)
    
    t0 = time.time()
    
    # 2. Calcul massif de la distance entre cet article et les 364 000 autres !
    if distance_metric == 'cosine':
        # Cosine Similarity : Plus on est proche de 1, plus c'est similaire
        similarities = cosine_similarity(target_vector, embeddings)[0]
        # On trie (argsort) par ordre décroissant et on prend le Top 5
        # (On ignore le 1er car c'est l'article lui-même avec un score de 1.0)
        top_indices = np.argsort(similarities)[::-1][1:top_n+1] 
        scores = similarities[top_indices]
        
    elif distance_metric == 'euclidean':
        # Euclidean Distance : Plus on est proche de 0, plus c'est similaire
        distances = euclidean_distances(target_vector, embeddings)[0]
        # On trie par ordre croissant (les plus petites distances)
        top_indices = np.argsort(distances)[1:top_n+1]
        scores = distances[top_indices]
        
    latency = (time.time() - t0) * 1000 # Conversion en ms
    
    return top_indices, scores, latency

print("✅ Moteur prêt à l'emploi !")


🎯 Création du moteur de recommandation...
✅ Moteur prêt à l'emploi !


In [3]:
# ==========================================
# BENCHMARK : COSINUS vs EUCLIDIENNE
# ==========================================

# 1. On choisit un article de test au hasard (par exemple l'article à l'index 1234)
ARTICLE_TEST_ID = 1234

# On regarde de quoi parle cet article grâce aux Metadata
categorie_test = articles_df.loc[ARTICLE_TEST_ID, 'category_id']

print(f"🧐 ARTICLE CIBLE : ID n°{ARTICLE_TEST_ID} (Catégorie : {categorie_test})")
print("=" * 60)

# --- TEST 1 : COSINUS ---
indices_cos, scores_cos, latence_cos = recommend_content_based(
    ARTICLE_TEST_ID, embeddings_pca, top_n=5, distance_metric='cosine'
)

print(f"\n📐 [1] SIMILARITÉ COSINUS (Temps de calcul : {latence_cos:.2f} ms)")
for i, (idx, score) in enumerate(zip(indices_cos, scores_cos)):
    cat = articles_df.loc[idx, 'category_id']
    match = "✅" if cat == categorie_test else "❌"
    print(f"   Top {i+1} : Article n°{idx:<6} | Catégorie {cat:<3} {match} | Score (proche de 1) : {score:.3f}")

# --- TEST 2 : EUCLIDIENNE ---
indices_euc, scores_euc, latence_euc = recommend_content_based(
    ARTICLE_TEST_ID, embeddings_pca, top_n=5, distance_metric='euclidean'
)

print(f"\n📏 [2] DISTANCE EUCLIDIENNE (Temps de calcul : {latence_euc:.2f} ms)")
for i, (idx, score) in enumerate(zip(indices_euc, scores_euc)):
    cat = articles_df.loc[idx, 'category_id']
    match = "✅" if cat == categorie_test else "❌"
    print(f"   Top {i+1} : Article n°{idx:<6} | Catégorie {cat:<3} {match} | Distance (proche de 0) : {score:.3f}")


🧐 ARTICLE CIBLE : ID n°1234 (Catégorie : 1)

📐 [1] SIMILARITÉ COSINUS (Temps de calcul : 74.92 ms)
   Top 1 : Article n°1335   | Catégorie 1   ✅ | Score (proche de 1) : 0.888
   Top 2 : Article n°993    | Catégorie 1   ✅ | Score (proche de 1) : 0.876
   Top 3 : Article n°347    | Catégorie 1   ✅ | Score (proche de 1) : 0.870
   Top 4 : Article n°1702   | Catégorie 1   ✅ | Score (proche de 1) : 0.869
   Top 5 : Article n°3944   | Catégorie 1   ✅ | Score (proche de 1) : 0.869

📏 [2] DISTANCE EUCLIDIENNE (Temps de calcul : 53.03 ms)
   Top 1 : Article n°1335   | Catégorie 1   ✅ | Distance (proche de 0) : 3.411
   Top 2 : Article n°993    | Catégorie 1   ✅ | Distance (proche de 0) : 3.428
   Top 3 : Article n°3944   | Catégorie 1   ✅ | Distance (proche de 0) : 3.529
   Top 4 : Article n°1587   | Catégorie 1   ✅ | Distance (proche de 0) : 3.665
   Top 5 : Article n°3947   | Catégorie 1   ✅ | Distance (proche de 0) : 3.726


In [4]:
# Chargement de l'historique des clics (Nécessaire pour le Hit Ratio)
CLICKS_PATH = os.path.join(GENERATED_DIR, "clicks_full.parquet")

print("📂 Chargement du dataset des clics...")
clicks_df = pd.read_parquet(CLICKS_PATH)
print(f"✅ Dataset chargé : {len(clicks_df)} interactions.")

📂 Chargement du dataset des clics...
✅ Dataset chargé : 2988181 interactions.


In [ ]:
# ==========================================
# EVALUATION GLOBALE : HIT RATIO, MRR & COVERAGE
# ==========================================
print("🛠️ Préparation des données d'évaluation...")

# 1. On isole les utilisateurs ayant au moins 2 clics
user_counts = clicks_df['user_id'].value_counts()
valid_users = user_counts[user_counts >= 2].index

df_valid = clicks_df[clicks_df['user_id'].isin(valid_users)].copy()
df_valid = df_valid.sort_values(['user_id', 'click_timestamp'])

# 2. Le clic CIBLE (Celui qu'on doit deviner)
test_set = df_valid.groupby('user_id').tail(1)

# 3. Le clic PRECEDENT (L'input de notre algorithme Content-Based)
train_set = df_valid.drop(test_set.index)
last_clicks_train = train_set.groupby('user_id').tail(1)

# 4. On prend un échantillon de 10000 utilisateurs (comme pour l'ALS)
sample_users = test_set['user_id'].sample(10000, random_state=42).values

# Variables de suivi
hits_cos = 0
hits_euc = 0
mrr_cos = 0
mrr_euc = 0
recommended_items_cos = set()
recommended_items_euc = set()
catalog_size = len(embeddings_pca)

print("⏳ Lancement de l'évaluation sur 10000 utilisateurs (Durée estimée : ~3 minutes)...")
t0_global = time.time()

for user in sample_users:
    # L'article précédent lu par l'utilisateur
    prev_article = last_clicks_train[last_clicks_train['user_id'] == user]['click_article_id'].values[0]

    # L'article cible qu'il a lu ensuite
    target_article = test_set[test_set['user_id'] == user]['click_article_id'].values[0]

    # --- Recommandations Cosinus ---
    recs_cos, _, _ = recommend_content_based(prev_article, embeddings_pca, top_n=5, distance_metric='cosine')
    recommended_items_cos.update(recs_cos)

    if target_article in recs_cos:
        hits_cos += 1
        rank = list(recs_cos).index(target_article) + 1
        mrr_cos += 1.0 / rank

    # --- Recommandations Euclidiennes ---
    recs_euc, _, _ = recommend_content_based(prev_article, embeddings_pca, top_n=5, distance_metric='euclidean')
    recommended_items_euc.update(recs_euc)

    if target_article in recs_euc:
        hits_euc += 1
        rank = list(recs_euc).index(target_article) + 1
        mrr_euc += 1.0 / rank

print("\n" + "="*50)
print("🏁 RÉSULTATS DE L'ÉVALUATION (CONTENT-BASED)")
print("="*50)

print("\n🎯 Pertinence Métier (Hit Ratio @ 5) :")
print(f" - Cosinus     : {(hits_cos / 10000) * 100:.2f} %")
print(f" - Euclidienne : {(hits_euc / 10000) * 100:.2f} %")

print("\n🎯 Précision du Classement (MRR @ 5) :")
print(f" - Cosinus     : {mrr_cos / 10000:.4f}")
print(f" - Euclidienne : {mrr_euc / 10000:.4f}")

print("\n🌐 Diversité (Coverage) :")
print(f" - Cosinus     : {(len(recommended_items_cos) / catalog_size) * 100:.2f} %")
print(f" - Euclidienne : {(len(recommended_items_euc) / catalog_size) * 100:.2f} %")

print(f"\n⏱️ Temps total d'évaluation : {time.time() - t0_global:.2f} secondes")


🛠️ Préparation des données d'évaluation...
⏳ Lancement de l'évaluation sur 10000 utilisateurs (Durée estimée : ~3 minutes)...

🏁 RÉSULTATS DE L'ÉVALUATION (CONTENT-BASED)

🎯 Pertinence Métier (Hit Ratio @ 5) :
 - Cosinus     : 1.03 %
 - Euclidienne : 1.05 %

🎯 Précision du Classement (MRR @ 5) :
 - Cosinus     : 0.0064
 - Euclidienne : 0.0062

🌐 Diversité (Coverage) :
 - Cosinus     : 2.83 %
 - Euclidienne : 2.84 %

⏱️ Temps total d'évaluation : 1158.44 secondes


# Test sur les métriques de consommation (Inférence Unitaire)

In [ ]:
import tracemalloc
import time
import sys

print("🔍 TEST D'INFÉRENCE UNITAIRE (1 Utilisateur)")
test_user = sample_users[0]
print(f"Utilisateur cible : {test_user}")

# L'article précédent lu par l'utilisateur
prev_article = last_clicks_train[last_clicks_train['user_id'] == test_user]['click_article_id'].values[0]
print(f"Article d'origine (input) : {prev_article}\n")

# --- Test Cosinus ---
print("--- Modèle Content-Based (Cosinus) ---")
tracemalloc.start()
t0 = time.time()

recs_cos, scores_cos, _ = recommend_content_based(prev_article, embeddings_pca, top_n=5, distance_metric='cosine')

latency_cos = (time.time() - t0) * 1000
current, peak_cos = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"⏱️ Latence : {latency_cos:.2f} ms")
print(f"📈 Peak Memory (RAM allouée pendant le calcul) : {peak_cos / 1024:.2f} Ko")

# --- Test Euclidienne ---
print("\n--- Modèle Content-Based (Euclidienne) ---")
tracemalloc.start()
t0 = time.time()

recs_euc, scores_euc, _ = recommend_content_based(prev_article, embeddings_pca, top_n=5, distance_metric='euclidean')

latency_euc = (time.time() - t0) * 1000
current, peak_euc = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"⏱️ Latence : {latency_euc:.2f} ms")
print(f"📈 Peak Memory (RAM allouée pendant le calcul) : {peak_euc / 1024:.2f} Ko")

# Empreinte globale (taille de la matrice)
print("\n💾 Empreinte Mémoire (RAM estimée pour le déploiement) :")
print(f"Poids des Embeddings PCA en RAM : {sys.getsizeof(embeddings_pca) / (1024):.2f} Ko")

🔍 TEST D'INFÉRENCE UNITAIRE (1 Utilisateur)
Utilisateur cible : 122708
Article d'origine (input) : 285648

--- Modèle Content-Based (Cosinus) ---
⏱️ Latence : 56.46 ms
📈 Peak Memory (RAM allouée pendant le calcul) : 76795.04 Ko

--- Modèle Content-Based (Euclidienne) ---
⏱️ Latence : 65.35 ms
📈 Peak Memory (RAM allouée pendant le calcul) : 4274.07 Ko

💾 Empreinte Mémoire (RAM estimée pour le déploiement) :
Poids des Embeddings PCA en RAM : 0.12 Ko
